# Lab 05 — Segmentación de clientes de Olist

Analítica de Datos · O. Pinzón

Este notebook aplica `src/analytics/segment.py` sobre los datos reales de Olist:
construye la tabla RFM por cliente, elige el número de clusters con el método del
codo y las métricas de validación interna, entrena el clustering final, proyecta
los segmentos a dos dimensiones con PCA y los perfila en términos de negocio.

Todo el cálculo vive en el módulo. Aquí solo se orquesta y se visualiza.

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

print("raiz del proyecto:", RAIZ)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from analytics import extract, segment

DATA_RAW = RAIZ / "data" / "raw"
DATA_PROCESSED = RAIZ / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PALETA = ["#3366CC", "#EE7733", "#009988", "#AA3377", "#997700", "#44AADD"]
TINTA = "#1a1a1a"
GRIS = "#8a8a8a"

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": GRIS,
        "axes.labelcolor": TINTA,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "text.color": TINTA,
        "xtick.color": GRIS,
        "ytick.color": GRIS,
        "grid.color": "#e8e8e8",
        "grid.linewidth": 0.8,
        "font.size": 10,
    }
)

print("datos crudos:", DATA_RAW)

## 1. Tabla RFM

Se cargan las tablas de Olist, se construye el dataset base y se reduce a las tres
columnas que necesita el RFM: el identificador real del cliente, la fecha del
pedido y su valor.

`construir_dataset_base()` nombra la columna de valor `ticket_total`; la celda la
renombra a `monto_pedido`, que es lo que espera `calcular_rfm`.

In [ ]:
tablas = extract.cargar_olist(str(DATA_RAW))
df = extract.construir_dataset_base(tablas)

COLUMNA_VALOR = "ticket_total" if "ticket_total" in df.columns else "precio_total"

pedidos = df[["customer_unique_id", "order_purchase_timestamp", COLUMNA_VALOR]].rename(
    columns={COLUMNA_VALOR: "monto_pedido"}
)
pedidos = pedidos.dropna(subset=["monto_pedido"])

print("pedidos:", pedidos.shape)
pedidos.head()

In [ ]:
fecha_referencia = pedidos["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print("fecha de referencia:", fecha_referencia.date())

rfm = segment.calcular_rfm(pedidos, fecha_referencia)

print("clientes unicos:", len(rfm))
rfm.head()

In [ ]:
rfm[["recencia", "frecuencia", "monto"]].describe().round(2)

### Nota sobre la dimensión Frecuencia

La celda siguiente cuenta cuántos clientes tienen más de un pedido. El resultado
condiciona la interpretación de todo el clustering: si la inmensa mayoría de los
clientes compró una sola vez, la Frecuencia es casi una constante y el modelo se
apoya en la práctica solo en Recencia y Monto.

In [ ]:
distribucion_frecuencia = rfm["frecuencia"].value_counts().sort_index()
recurrentes = int((rfm["frecuencia"] > 1).sum())

print(f"clientes totales:        {len(rfm):,}")
print(f"clientes con frecuencia > 1: {recurrentes:,} ({recurrentes / len(rfm):.2%})")
print(f"varianza de frecuencia:  {rfm['frecuencia'].var():.4f}")
print()
print("distribucion de pedidos por cliente:")
print(distribucion_frecuencia.head(10).to_string())

## 2. Elección de k

Se prueba k de 2 a 8. Para cada valor se registran tres indicadores:

- **inercia** — suma de distancias cuadradas al centroide. Siempre baja al subir k,
  así que se lee buscando el codo, no el mínimo.
- **silhouette** — qué tan separados quedan los clusters, de -1 a 1. Más alto es mejor.
- **Davies-Bouldin** — razón entre dispersión interna y separación entre clusters.
  Más bajo es mejor.

In [ ]:
X = rfm[["recencia", "frecuencia", "monto"]]

resultados = []
for k in range(2, 9):
    pipeline_k = segment.entrenar_pipeline_clustering(X, n_clusters=k)
    etiquetas_k = pipeline_k.predict(X)
    metricas_k = segment.evaluar_clustering(X, etiquetas_k)
    resultados.append(
        {
            "k": k,
            "inercia": pipeline_k.named_steps["modelo"].inertia_,
            "silhouette": metricas_k["silhouette"],
            "davies_bouldin": metricas_k["davies_bouldin"],
        }
    )

tabla_k = pd.DataFrame(resultados).set_index("k")
tabla_k.round(4)

In [ ]:
k_silhouette = int(tabla_k["silhouette"].idxmax())
k_davies = int(tabla_k["davies_bouldin"].idxmin())

print(f"k con mejor silhouette:      {k_silhouette}")
print(f"k con mejor Davies-Bouldin:  {k_davies}")

In [ ]:
figura, (izq, der) = plt.subplots(1, 2, figsize=(11, 4))

izq.plot(
    tabla_k.index,
    tabla_k["inercia"],
    color=PALETA[0],
    linewidth=2,
    marker="o",
    markersize=8,
)
izq.set_title("Metodo del codo", loc="left", fontsize=12)
izq.set_xlabel("k")
izq.set_ylabel("inercia")
izq.grid(axis="y")
izq.set_axisbelow(True)

der.plot(
    tabla_k.index,
    tabla_k["silhouette"],
    color=PALETA[1],
    linewidth=2,
    marker="o",
    markersize=8,
    label="silhouette (mayor es mejor)",
)
der.plot(
    tabla_k.index,
    tabla_k["davies_bouldin"],
    color=PALETA[2],
    linewidth=2,
    marker="s",
    markersize=8,
    label="Davies-Bouldin (menor es mejor)",
)
der.axvline(k_silhouette, color=GRIS, linewidth=1, linestyle="--")
der.set_title("Validacion interna", loc="left", fontsize=12)
der.set_xlabel("k")
der.grid(axis="y")
der.set_axisbelow(True)
der.legend(frameon=False, fontsize=9)

figura.tight_layout()
plt.show()

### Justificación de la elección

**Criterio aplicado.** Las dos curvas no siempre coinciden, y cuando eso ocurre hay
que decidir con un argumento, no con el número más alto:

- El **silhouette** premia la separación geométrica. Sobre datos RFM muy
  asimétricos tiende a favorecer valores bajos de k, porque un corte grueso entre
  "clientes normales" y "clientes de gasto extremo" produce grupos muy separados
  aunque no sean accionables.
- El **codo** marca el punto donde agregar un cluster más deja de reducir la
  inercia de forma apreciable.
- El tercer criterio, que no es estadístico, es la **accionabilidad**: un
  segmento solo sirve si el negocio puede hacer algo distinto con él.

**Decisión.** Se elige el k de la celda siguiente. Cuando el silhouette sugiere un k
menor que el codo, se prefiere el valor que produce segmentos con perfiles de
negocio distinguibles, siempre que su silhouette no caiga de forma drástica
respecto al máximo. Un clustering con el mejor silhouette pero que solo separa
outliers de gasto no responde la pregunta de negocio que motiva el RFM.

> Ajustar `K_ELEGIDO` en la celda de abajo según la tabla y las curvas, y completar
> aquí la justificación con los valores observados.

In [ ]:
K_ELEGIDO = k_silhouette

pipeline = segment.entrenar_pipeline_clustering(X, n_clusters=K_ELEGIDO)
rfm["segmento"] = pipeline.predict(X)

metricas = segment.evaluar_clustering(X, rfm["segmento"])

print(f"k elegido: {K_ELEGIDO}")
for clave, valor in metricas.items():
    print(f"  {clave}: {valor:.4f}")
print()
print("clientes por segmento:")
print(rfm["segmento"].value_counts().sort_index().to_string())

## 3. Visualización PCA

Las tres dimensiones del RFM se proyectan a dos componentes principales para poder
ver la separación. La proyección escala antes de reducir, igual que el pipeline de
clustering, para que ninguna dimensión domine por su unidad.

In [ ]:
coordenadas = segment.reducir_a_2d(X)

figura, ejes = plt.subplots(figsize=(7.5, 6))

for indice, etiqueta in enumerate(sorted(rfm["segmento"].unique())):
    mascara = rfm["segmento"] == etiqueta
    ejes.scatter(
        coordenadas[mascara, 0],
        coordenadas[mascara, 1],
        s=9,
        alpha=0.5,
        color=PALETA[indice % len(PALETA)],
        label=f"Segmento {etiqueta} (n={int(mascara.sum()):,})",
        linewidths=0,
    )

ejes.set_title(
    f"Segmentos RFM proyectados con PCA (k={K_ELEGIDO})", loc="left", fontsize=12
)
ejes.set_xlabel("componente principal 1")
ejes.set_ylabel("componente principal 2")
ejes.grid(True)
ejes.set_axisbelow(True)
ejes.legend(frameon=False, fontsize=9, markerscale=2)

figura.tight_layout()
plt.show()

## 4. Perfilado de los segmentos

`perfilar_segmentos` devuelve el promedio de cada dimensión RFM por cluster, junto
con el número de clientes. Es la tabla que traduce el resultado geométrico en algo
que el negocio puede leer.

In [ ]:
perfil = segment.perfilar_segmentos(
    rfm,
    columna_segmento="segmento",
    columnas_perfil=["recencia", "frecuencia", "monto"],
)

perfil.round(2)

In [ ]:
perfil_relativo = perfil[["recencia", "frecuencia", "monto"]].copy()
for columna in perfil_relativo.columns:
    perfil_relativo[columna] = perfil_relativo[columna] / rfm[columna].mean()

print("perfil como multiplo del promedio global (1.00 = igual al promedio):")
perfil_relativo.round(2)

### Nombres y acciones de negocio propuestas

La lectura se hace sobre la tabla relativa de arriba: un valor mayor que 1 indica
que el segmento está por encima del promedio global en esa dimensión. Para
Recencia, estar por encima del promedio es malo — significa que hace más tiempo
que no compran.

| Segmento | Perfil observado | Nombre propuesto | Acción de negocio |
|---|---|---|---|
| — | recencia baja, monto alto | Clientes de alto valor | Programa de fidelización, acceso anticipado a lanzamientos, soporte prioritario |
| — | recencia baja, monto bajo | Clientes recientes de ticket bajo | Recomendación cruzada y envío gratis sobre un mínimo, para subir el ticket |
| — | recencia alta, monto alto | En riesgo de fuga | Campaña de reactivación con descuento agresivo; es el segmento donde más se pierde por inacción |
| — | recencia alta, monto bajo | Inactivos de bajo valor | No invertir en reactivación individual; incluir solo en campañas masivas de bajo costo |

> Completar la primera columna con el número de cluster que corresponde a cada
> perfil, según la tabla relativa. Los números de cluster que asigna K-Means no
> tienen orden ni significado propio: cambian entre corridas con distinta semilla.

## 5. Detección de anomalías con DBSCAN

DBSCAN no asume un número de grupos: marca como `-1` todo punto que no pertenezca a
una región suficientemente densa. Sobre el RFM escalado, esos puntos son candidatos
a gasto atípico.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_escalado = StandardScaler().fit_transform(X)

EPS = 0.5
MIN_SAMPLES = 10

etiquetas_dbscan = segment.detectar_anomalias(
    X_escalado, eps=EPS, min_samples=MIN_SAMPLES
)

rfm["anomalia_dbscan"] = etiquetas_dbscan == -1
n_anomalias = int(rfm["anomalia_dbscan"].sum())

print(f"eps={EPS}, min_samples={MIN_SAMPLES}")
print(f"anomalias (-1): {n_anomalias:,} ({n_anomalias / len(rfm):.2%})")
n_grupos = len(set(etiquetas_dbscan)) - (1 if -1 in etiquetas_dbscan else 0)
print(f"grupos densos encontrados: {n_grupos}")

In [ ]:
print("perfil de los clientes marcados como anomalia:")
anomalias = rfm[rfm["anomalia_dbscan"]]
print(anomalias[["recencia", "frecuencia", "monto"]].describe().round(2))
print()
print("los 10 montos mas altos entre las anomalias:")
columnas = ["customer_unique_id", "recencia", "frecuencia", "monto"]
print(anomalias.nlargest(10, "monto")[columnas].to_string(index=False))

> Revisar la salida anterior para responder 9.4: si los clientes marcados tienen
> montos de varios múltiplos del promedio o frecuencias inusualmente altas, se trata
> de gasto genuinamente atípico. Si el porcentaje marcado es muy alto (más de un
> 10–15% del total), el `eps` está demasiado bajo para la densidad real de los datos
> y conviene subirlo antes de interpretar el resultado.

## 6. Guardado del resultado

El dataset segmentado se guarda en `data/processed/` para que las etapas
posteriores del pipeline lo consuman sin repetir el entrenamiento.

In [ ]:
salida = DATA_PROCESSED / "clientes_segmentados.csv"
rfm.to_csv(salida, index=False)

print("guardado en:", salida)
print("filas:", len(rfm))
print("columnas:", list(rfm.columns))